PRÉPARATION DES DONNÉES

In [2]:
import pandas as pd

In [5]:
df = pd.read_csv('/content/sample_data/Tweets.csv')
df = df[['text', 'airline_sentiment']]

Nettoyage texte

In [8]:
stopwords = ['i'] + ['a'] + ['an'] + ['the']
mentions = '@VirginAmerica'

remove_stopwords = lambda x: ' '.join([word for word in x.split() if word not in (stopwords)])
remove_mentions = lambda x: ' '.join([word for word in x.split() if not word.startswith('@')])

df.text = df.text.apply(remove_stopwords).apply(remove_mentions)

Split données

In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df.text, df.airline_sentiment,
    test_size=0.1,
    random_state=42
)

TOKENIZATION (conversion texte vers nombres)

In [11]:
from tensorflow.keras.preprocessing.text import Tokenizer


tk = Tokenizer(num_words=10000,
               filters='!"#$%&()*+,-./:;<=>?@[\\]^_`{|}~\t\n',
               lower=True,
               split=' ')

In [ ]:
tk.fit_on_texts(X_train)

Conversion en matrice binaire

In [12]:
X_train_oh = tk.texts_to_matrix(X_train, mode='binary')
X_test_oh = tk.texts_to_matrix(X_test, mode='binary')

LABEL ENCODING

In [13]:
from sklearn.preprocessing import LabelEncoder
from keras.utils import to_categorical

le = LabelEncoder()

y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

y_train = to_categorical(y_train)
y_test = to_categorical(y_test)

VALIDATION SET

In [14]:
X_train_rest, X_valid, y_train_rest, y_valid = train_test_split(
    X_train_oh, y_train,
    test_size=0.1,
    random_state=37
)

MODÈLE DE BASE (OVERFITTING)

In [15]:
from keras import models, layers

base_model = models.Sequential()

base_model.add(layers.Dense(64, activation='relu', input_shape=(10000,)))
base_model.add(layers.Dense(64, activation='relu'))
base_model.add(layers.Dense(3, activation='softmax'))

base_model._name = "Base_Model"

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


TRAINING FUNCTION

In [16]:
def deep_model(model, X_train, y_train, X_valid, y_valid):
    model.compile(optimizer='rmsprop',
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])

    history = model.fit(
        X_train, y_train,
        epochs=20,
        batch_size=512,
        validation_data=(X_valid, y_valid),
        verbose=0
    )

    return history

OVERFITTING OBSERVATION

In [17]:
base_history = deep_model(base_model, X_train_rest, y_train_rest, X_valid, y_valid)

SOLUTIONS : 1/L2 Regularization

In [18]:
from keras import regularizers

reg_model = models.Sequential()

reg_model.add(layers.Dense(64,
                           activation='relu',
                           kernel_regularizer=regularizers.l2(0.001),
                           input_shape=(10000,)))

reg_model.add(layers.Dense(64,
                           activation='relu',
                           kernel_regularizer=regularizers.l2(0.001)))

reg_model.add(layers.Dense(3, activation='softmax'))

2/L1 Regularization

In [19]:
reg_model_L1 = models.Sequential()

reg_model_L1.add(layers.Dense(64,
                              activation='relu',
                              kernel_regularizer=regularizers.l1(0.001),
                              input_shape=(10000,)))

reg_model_L1.add(layers.Dense(64,
                              activation='relu',
                              kernel_regularizer=regularizers.l1(0.001)))

reg_model_L1.add(layers.Dense(3, activation='softmax'))

3/Dropout

In [20]:
drop_model = models.Sequential()

drop_model.add(layers.Dense(64, activation='relu', input_shape=(10000,)))
drop_model.add(layers.Dropout(0.5))

drop_model.add(layers.Dense(64, activation='relu'))
drop_model.add(layers.Dropout(0.5))

drop_model.add(layers.Dense(3, activation='softmax'))

Comparaison

In [25]:
compare_models_by_metric = lambda base_model, model, base_history, history, metric: print('model accuracy:', model.evaluate(X_test_oh, y_test)[1])
reg_history = deep_model(reg_model, X_train_rest, y_train_rest, X_valid, y_valid)
reg_history_L1 = deep_model(reg_model_L1, X_train_rest, y_train_rest, X_valid, y_valid)
drop_history = deep_model(drop_model, X_train_rest, y_train_rest, X_valid, y_valid)


compare_models_by_metric(base_model, reg_model, base_history, reg_history, 'val_loss')
compare_models_by_metric(base_model, reg_model_L1, base_history, reg_history_L1, 'val_loss')
compare_models_by_metric(base_model, drop_model, base_history, drop_history, 'val_loss')

46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6311 - loss: 0.9248
model accuracy: 0.631147563457489
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6311 - loss: 1.1536
model accuracy: 0.631147563457489
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6311 - loss: 0.9246
model accuracy: 0.631147563457489
